Looking at Autoantibody results from ImmPort Studies
* SDY1737
* SDY524
* SDY569

* SDY1625
* SDY655 ?
* SDY797 ?
* SDY824
* SDY91

Look at K-Means 
Unsupervised clustering

In [1]:
# === Package setup ===
#
# Install only the packages this notebook actually uses, in one pass.
# Replaces several earlier setup cells, including a malformed `require()`
# check that triggered a slow `devtools::install_github` every run and an
# unnecessary route of `dplyr` through BiocManager (`dplyr` is a CRAN package).
#
# Packages used downstream by the analysis cells:
#   tidyverse  -- bundle: dplyr, tidyr, tibble, stringr, ggplot2, readr, purrr
#   lubridate  -- date handling
#   tidyHeatmap, ComplexHeatmap -- heatmap visualisation
# All from CRAN except ComplexHeatmap (Bioconductor).

options(repos = c(CRAN = "https://cloud.r-project.org"))

cran_pkgs <- c("tidyverse", "lubridate", "tidyHeatmap")
for (p in cran_pkgs) {
    if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
}

if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if (!requireNamespace("ComplexHeatmap", quietly = TRUE))
    BiocManager::install("ComplexHeatmap", update = FALSE, ask = FALSE)

suppressPackageStartupMessages({
    library(tidyverse)
    library(lubridate)
    library(tidyHeatmap)
    library(ComplexHeatmap)
})

cat("Package setup complete.\n")
cat("  Loaded: tidyverse (dplyr/tidyr/tibble/stringr/ggplot2/readr/purrr),\n")
cat("          lubridate, tidyHeatmap, ComplexHeatmap.\n")


also installing the dependencies ‘sys’, ‘askpass’, ‘gargle’, ‘ids’, ‘openssl’, ‘textshaping’, ‘googledrive’, ‘googlesheets4’, ‘httr’, ‘ragg’, ‘reprex’, ‘rstudioapi’, ‘rvest’


Warning message in install.packages(p):
“installation of package ‘sys’ had non-zero exit status”
Warning message in install.packages(p):
“installation of package ‘textshaping’ had non-zero exit status”
Warning message in install.packages(p):
“installation of package ‘rstudioapi’ had non-zero exit status”
Warning message in install.packages(p):
“installation of package ‘askpass’ had non-zero exit status”
Warning message in install.packages(p):
“installation of package ‘ragg’ had non-zero exit status”
Warning message in install.packages(p):
“installation of package ‘reprex’ had non-zero exit status”
Warning message in install.packages(p):
“installation of package ‘openssl’ had non-zero exit status”
Warning message in install.packages(p):
“installation of package ‘ids’ had non-zero exit status”
Warning message in in

ERROR: Error in library(tidyverse): there is no package called ‘tidyverse’


**Function:** *make_aa_heatmaps* 

**Purpose** 

*make_aa_heatmaps* will make as requested a k-means heatmap - clustering by row with k=4 and clustering by autoantboid k = 2.   As generizable as we can given we know the autoantibodies and the heatmaps we want to make.  

In [ ]:
# ── minimal deps
library(dplyr)
library(tidyr)
library(tidyHeatmap)
library(ComplexHeatmap)

# ───────────────────────────────────────────────────────────────────────────────
# 1) Small utilities
# ───────────────────────────────────────────────────────────────────────────────
aa_age_to_group <- function(age) {
  grp <- dplyr::case_when(
    is.na(age)    ~ "Missing",
    age < 8       ~ "8-12",
    age <= 12     ~ "8-12",
    age <= 17     ~ "13-17",
    TRUE          ~ "18-30"
  )
  factor(grp, levels = c("8-12","13-17","18-30","Missing"))
}

aa_first_col <- function(df, candidates) {
  hit <- intersect(candidates, names(df))
  if (length(hit)) hit[[1]] else NULL
}

In [ ]:
# ───────────────────────────────────────────────────────────────────────────────
# 2) Canonicalize + (optionally) restrict visits + longify to Property/Value
#    Works for both formats:
#      - long:   analyte_name_col + analyte_value_col present (e.g., SDY524)
#      - wide:   analytes are columns (e.g., SDY1737)
# ───────────────────────────────────────────────────────────────────────────────
aa_prepare <- function(
  df,
  accession_col,
  sex_col,
  age_group_col = NULL,         # if not given, compute from age_years_col
  age_years_col = NULL,
  analyte_name_col = NULL,      # (long) e.g., "AutoAB"
  analyte_value_col = NULL,     # (long) e.g., "Value"
  analyte_cols = NULL,          # (wide) e.g., c("GAD65","IA_2ic","MIAA","Zn_T8")
  visit_cols = c("Visit_Num","Visitnum","Visit"),
  restrict_visits = NULL        # numeric or -1 (for textual baseline)
) {
  stopifnot(accession_col %in% names(df), sex_col %in% names(df))

  out <- df %>%
    dplyr::rename(
      Accession = !!rlang::sym(accession_col),
      Sex       = !!rlang::sym(sex_col)
    )

  # ---- Age grouping (canonical labels)
  AGE_LEVELS <- c("0-7","8-12","13-17","18-30",">30","Missing")
    
    # replace aa_age_to_group() inside aa_prepare() with:
    aa_age_to_group <- function(x) {
        xx <- trimws(as.character(x))
        xx[xx == ""] <- NA_character_
        # map pre-labeled buckets safely (no "" key)
        map <- c(
            "0-7" = "0-7", "<8" = "0-7", "< 8" = "0-7",
            "8-12" = "8-12",
            "13-17" = "13-17",
            "18-30" = "18-30",
            ">30" = ">30", "> 30" = ">30", "30+" = ">30", "31+" = ">30",
            "Missing" = "Missing", "missing" = "Missing"
        )
        lbl <- unname(map[xx])  # unmatched → NA
        # numeric fallback for anything not matched above
        num  <- suppressWarnings(as.numeric(xx))
        fill <- is.na(lbl) & !is.na(num)
        lbl[fill & num < 8]               <- "0-7"
        lbl[fill & num >= 8  & num <= 12] <- "8-12"
        lbl[fill & num >= 13 & num <= 17] <- "13-17"
        lbl[fill & num >= 18 & num <= 30] <- "18-30"
        lbl[fill & num > 30]              <- ">30"
        lbl[is.na(lbl)] <- "Missing"
        
        factor(lbl, levels = c("0-7","8-12","13-17","18-30",">30","Missing"))
    }
    
    if (!is.null(age_group_col) && age_group_col %in% names(out)) {
        out <- out %>% dplyr::mutate(Age_Group = aa_age_to_group(.data[[age_group_col]]))
    } else if (!is.null(age_years_col) && age_years_col %in% names(out)) {
        out <- out %>% dplyr::mutate(Age_Group = aa_age_to_group(.data[[age_years_col]]))
    } else {
        out <- out %>% dplyr::mutate(Age_Group = factor("Missing", levels = AGE_LEVELS))
    }
    
# ---- Visit filter (numeric or textual baseline)
vcol <- intersect(visit_cols, names(out))[1]
if (!is.na(vcol) && length(vcol) && !is.null(restrict_visits)) {
  vraw <- out[[vcol]]
  vnum <- suppressWarnings(as.numeric(as.character(vraw)))

  if (!all(is.na(vnum))) {
    # numeric/labelled visits
    keep <- !is.na(vnum) & vnum %in% restrict_visits
    out  <- out[keep, , drop = FALSE]
    out[[vcol]] <- vnum[keep]   # <-- align length with filtered rows
  } else {
    # textual visits: allow only baseline via -1
    if (all(restrict_visits == -1)) {
      vtxt <- tolower(trimws(as.character(vraw)))
      base_syn <- c("baseline","screening","day -1","day-1","v-1","-1")
      keep <- vtxt %in% base_syn
      out  <- out[keep, , drop = FALSE]
    } else {
      stop("Textual visit column cannot match non-baseline visits.")
    }
  }
}

    
  # ---- Long input
  if (!is.null(analyte_name_col) && !is.null(analyte_value_col) &&
      analyte_name_col %in% names(out) && analyte_value_col %in% names(out)) {

    tidy <- out %>%
      dplyr::transmute(
        Accession, Sex, Age_Group,
        Property = .data[[analyte_name_col]],
        Value    = suppressWarnings(as.numeric(.data[[analyte_value_col]]))
      )

  } else {
    # ---- Wide input → long
    id_like <- c("Accession","Sex","Age_Group", visit_cols, age_years_col,
                 "Cohort_Group","Baseline_Height_cm","Baseline_Weight_kg","Baseline_BMI_kg_m2")

    # (A) If analyte_cols explicitly provided, use them and force numeric
    if (!is.null(analyte_cols)) {
      present <- intersect(analyte_cols, names(out))
      if (!length(present)) stop("None of the specified analyte_cols are present.")
      out[present] <- lapply(out[present], function(x) suppressWarnings(as.numeric(x)))
      value_cols <- present
    } else {
      # (B) Coerce common antibody names if present
      common <- intersect(c("GAD65","IA_2ic","MIAA","Zn_T8"), names(out))
      if (length(common)) {
        out[common] <- lapply(out[common], function(x) suppressWarnings(as.numeric(x)))
      }
      num_cols   <- names(out)[vapply(out, is.numeric, logical(1))]
      id_present <- intersect(id_like, names(out))
      value_cols <- setdiff(num_cols, id_present)
    }

    if (!length(value_cols))
      stop('No numeric analyte columns to melt in wide→long step. Supply analyte_cols=c("GAD65","IA_2ic","MIAA","Zn_T8").')

    tidy <- out %>%
      dplyr::select(Accession, Sex, Age_Group, dplyr::all_of(value_cols)) %>%
      tidyr::pivot_longer(dplyr::all_of(value_cols), names_to = "Property", values_to = "Value")
  }

  if (!nrow(tidy)) stop("No rows left after visit filtering/preparation.")

  tidy %>%
    dplyr::group_by(Accession, Sex, Age_Group, Property) %>%
    dplyr::summarise(Value = suppressWarnings(mean(Value, na.rm = TRUE)), .groups = "drop")
}

In [ ]:
# ───────────────────────────────────────────────────────────────────────────────
# 3) Build matrix (+ simple NA fill for clustering only)
# ───────────────────────────────────────────────────────────────────────────────

aa_build_matrix <- function(tidy_df) {
  tidy_df %>%
    filter(!is.na(Value), !is.na(Property)) %>%
    select(Accession, Property, Value) %>%  # ✅ Only include necessary cols
    tidyr::pivot_wider(names_from = Property, values_from = Value) %>%
    tibble::column_to_rownames("Accession") %>%
    {
      mat <- as.matrix(.)
      storage.mode(mat) <- "double"
      list(mat = mat)
    }
}


In [ ]:
aa_fill_for_clustering <- function(mat) {
  if (!nrow(mat) || !ncol(mat)) return(mat)
  # column medians
  cm <- apply(mat, 2, function(x) {
    m <- suppressWarnings(stats::median(x, na.rm = TRUE))
    if (is.infinite(m) || is.na(m)) 0 else m
  })
  # fill NAs
  for (j in seq_len(ncol(mat))) {
    na_idx <- is.na(mat[, j])
    if (any(na_idx)) mat[na_idx, j] <- cm[j]
  }
  mat
}

In [ ]:
# ───────────────────────────────────────────────────────────────────────────────
# 4) Two small plotters: ward.D2 and k-means (kmeans done outside to avoid NA issues)
# ───────────────────────────────────────────────────────────────────────────────
aa_plot_ward <- function(tidy_df, scale_mode = "column",
                         dist_method = "minkowski", minkowski_p = 2) {
  dist_fun <- function(x) stats::dist(x, method = dist_method, p = minkowski_p)

  hm <- tidy_df %>%
    tidyHeatmap::heatmap(
      .row    = Accession,
      .column = Property,
      .value  = Value,
      scale   = scale_mode,
      cluster_rows = TRUE,
      cluster_columns = TRUE,
      clustering_distance_rows    = dist_fun,
      clustering_method_rows      = "ward.D2",
      clustering_distance_columns = dist_fun,
      clustering_method_columns   = "ward.D2"
    )

  if ("Sex" %in% names(tidy_df))       hm <- hm %>% annotation_tile(Sex)
  if ("Age_Group" %in% names(tidy_df)) hm <- hm %>% annotation_tile(Age_Group)
  # Modify within aa_plot_kmeans() and aa_plot_ward()
  if ("Study" %in% names(tidy_df)) hm <- hm %>% annotation_tile(Study)

  hm
}

In [ ]:
aa_plot_kmeans <- function(tidy_df, km_rows = 4, km_cols = 2, scale_mode = "none") {
  mm <- aa_build_matrix(tidy_df)
  if (!nrow(mm$mat) || !ncol(mm$mat)) stop("Empty matrix for k-means heatmap.")

  mat_fill <- aa_fill_for_clustering(mm$mat)

  # safe k
  kr <- max(1, min(km_rows, nrow(mat_fill)))
  kc <- max(1, min(km_cols, ncol(mat_fill)))

  # robust kmeans (fallback to 1 cluster if it errors)
  kmeans_safe <- function(x, centers) {
    out <- try(stats::kmeans(x, centers = centers, iter.max = 50), silent = TRUE)
    if (inherits(out, "try-error")) list(cluster = rep(1L, nrow(x))) else out
  }

  set.seed(1)
  rkm <- kmeans_safe(mat_fill,  kr)
  ckm <- kmeans_safe(t(mat_fill), kc)

  row_lab <- paste0("R", as.integer(rkm$cluster))
  col_lab <- paste0("C", as.integer(ckm$cluster))

  # named split vectors aligned to the matrix order
  row_split_vec <- factor(row_lab, levels = unique(row_lab))
  names(row_split_vec) <- rownames(mat_fill)
  row_split_vec <- row_split_vec[rownames(mm$mat)]

  col_split_vec <- factor(col_lab, levels = unique(col_lab))
  names(col_split_vec) <- colnames(mat_fill)
  col_split_vec <- col_split_vec[colnames(mm$mat)]

  hm <-  tibble::as_tibble(tidy_df) %>%
    tidyHeatmap::heatmap(
      .row         = Accession,
      .column      = Property,
      .value       = Value,
      scale        = scale_mode,
      row_split    = row_split_vec,
      column_split = col_split_vec,
      cluster_rows = TRUE,
      cluster_columns = TRUE
#      row_order    = rownames(mm$mat),
#      column_order = colnames(mm$mat)
    )

  if ("Sex" %in% names(tidy_df))       hm <- hm %>% tidyHeatmap::annotation_tile(Sex)
  if ("Age_Group" %in% names(tidy_df)) hm <- hm %>% tidyHeatmap::annotation_tile(Age_Group)

  # Modify within aa_plot_kmeans() and aa_plot_ward()
  if ("Study" %in% names(tidy_df)) hm <- hm %>% annotation_tile(Study)


  hm
}

In [ ]:
# ───────────────────────────────────────────────────────────────────────────────
# 5) Tiny wrapper
# ───────────────────────────────────────────────────────────────────────────────
aa_heatmaps <- function(tidy_df,
                        scale_mode = "column",
                        dist_method = "minkowski", minkowski_p = 2,
                        k_rows = 4, k_cols = 2) {
  list(
    hm_ward   = aa_plot_ward(tidy_df, scale_mode, dist_method, minkowski_p),
    hm_kmeans = aa_plot_kmeans(tidy_df, k_rows, k_cols, scale_mode)
  )
}

**ImmPort Study 1737**

Note that subject "SUB228869" had 3 visits, we kept only the first one  - manually removed from the file

In [ ]:
# ========== STEP 0: Set working directory (for running locally on laptop) =========
from __future__ import annotations
import sys, os, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
print("Repo:", REPO)

In [ ]:
sdy1737_antibody <- as.matrix(read.csv("data/SDY1737/AutoAB_2021-02-16_13-46-53_ITN041AI.csv",sep=","))

In [ ]:
sdy1737_antibody.df <- data.frame(sdy1737_antibody)

In [ ]:
head (sdy1737_antibody)

In [ ]:
sdy1737_subj <- as.matrix(read.csv("data/SDY1737/ADAE1_2021-02-16_13-46-25_ITN041AI.csv",sep=","))

In [ ]:
sdy1737_subj.df <- data.frame(sdy1737_subj)

In [ ]:
colnames(sdy1737_subj.df)

In [ ]:
    sdy1737_subj.df %>%
      group_by(Sex_Char) %>%
      summarise(count = n()) %>%
      arrange(desc(count))

In [ ]:
sdy1737_subj.df %>%
      group_by(Cohort_Group) %>%
      summarise(count = n()) %>%
      arrange(desc(count))

In [ ]:
sdy1737_subj.df %>%
  summarise(
    mean_var = mean(as.numeric(Age_years), na.rm = TRUE),
    median_var = median(as.numeric(Age_years), na.rm = TRUE),
    sd_var = sd(Age_years, na.rm = TRUE),
    min_var = min(Age_years, na.rm = TRUE),
    max_var = max(Age_years, na.rm = TRUE)
  )

In [ ]:
sdy1737_antibody.df %>%
  summarise(
    mean_var = mean(as.numeric(GAD65), na.rm = TRUE),
    median_var = median(as.numeric(GAD65), na.rm = TRUE),
    sd_var = sd(GAD65, na.rm = TRUE),
    min_var = min(GAD65, na.rm = TRUE),
    max_var = max(GAD65, na.rm = TRUE),
    count = n(), # Total count of rows,
    non_na_count = sum(!is.na(GAD65)) # Count of non-NA values in 'value'
  )

In [ ]:
sdy1737_antibody.df %>%
  summarise(
    mean_var = mean(as.numeric(IA_2ic), na.rm = TRUE),
    median_var = median(as.numeric(IA_2ic), na.rm = TRUE),
    sd_var = sd(IA_2ic, na.rm = TRUE),
    min_var = min(IA_2ic, na.rm = TRUE),
    max_var = max(IA_2ic, na.rm = TRUE),
    count = n(), # Total count of rows,
    non_na_count = sum(!is.na(IA_2ic)) # Count of non-NA values in 'value'
  )

In [ ]:
sdy1737_antibody.df %>%
  summarise(
    mean_var = mean(as.numeric(MIAA), na.rm = TRUE),
    median_var = median(as.numeric(MIAA), na.rm = TRUE),
    sd_var = sd(MIAA, na.rm = TRUE),
    min_var = min(MIAA, na.rm = TRUE),
    max_var = max(MIAA, na.rm = TRUE),
    count = n(), # Total count of rows,
    non_na_count = sum(!is.na(MIAA)) # Count of non-NA values in 'value'
  )

In [ ]:
sdy1737_antibody.df %>%
  summarise(
    mean_var = mean(as.numeric(Zn_T8), na.rm = TRUE),
    median_var = median(as.numeric(Zn_T8), na.rm = TRUE),
    sd_var = sd(Zn_T8, na.rm = TRUE),
    min_var = min(Zn_T8, na.rm = TRUE),
    max_var = max(Zn_T8, na.rm = TRUE),
    count = n(), # Total count of rows,
    non_na_count = sum(!is.na(Zn_T8)) # Count of non-NA values in 'value'
  )

We want to combine the clinical features with the autoantibody results -- we will do this with a left join by Accesssion which is the subject ID but there were multiple visits and so what I really want is to create a unique table for joining the items contained will be Accession, Sex_char, Age_years, Cohort_Group, Baseline_Height_cm, Baseline_Weight_kg, Baseline_BMI_kg_m2


In [ ]:
subject_df <- data.frame(unique(sdy1737_subj.df[, c("Accession", 
                                  "Sex_Char", 
                                  "Age_years", 
                                  "Cohort_Group", 
                                  "Baseline_Height_cm", 
                                  "Baseline_Weight_kg", 
                                  "Baseline_BMI_kg_m2")]))

In [ ]:
sdy1737_df <- left_join(subject_df, sdy1737_antibody.df, by = "Accession")

In [ ]:
colnames(sdy1737_df)

In [ ]:
any(duplicated(names(sdy1737_df)))
names(sdy1737_df)[duplicated(names(sdy1737_df))]

In [ ]:
head(sdy1737_df[,c("Accession","GAD65","IA_2ic","MIAA","Zn_T8")])

The heatmaps visualise a multi-element, multi-feature dataset, annotated with independent variables. Each observation is a element-feature pair (e.g., person-physical characteristics).

element	feature	value	independent_variables
chr or fctr	chr or fctr	numeric	…
Let’s transform the mtcars dataset into a tidy “element-feature-independent variables” data frame. Where the independent variables in this case are ‘hp’ and ‘vs’.

In [ ]:
tidy_1737_v1 <- aa_prepare(
  sdy1737_df,
  accession_col   = "Accession",
  sex_col         = "Sex_Char",
  age_years_col   = "Age_years",                       # we compute Age_Group
  analyte_cols    = c("GAD65","IA_2ic","MIAA","Zn_T8"),# <- tell it the antibody columns
  visit_cols      = c("Visitnum","Visit"),
  restrict_visits = -1                              # baseline only
)

In [ ]:
hm_1737_v1 <- aa_heatmaps(
  tidy_1737_v1,
  scale_mode = "column",
  k_rows = 4,
  k_cols = 2
)

In [ ]:
hm_1737_v1$hm_ward

In [ ]:
hm_1737_v1$hm_kmeans

**SDY524** - longitudinal data


In [ ]:
getwd()

In [ ]:
sdy524_antibody <- as.matrix(read.csv("data/SDY524/SDY524_ITN-27AI.DataSet.Auto_Antibody.csv"))

In [ ]:
sdy524_antibody.df <- data.frame(sdy524_antibody)

In [ ]:
head(sdy524_antibody.df)

In [ ]:
sdy524_antibody.df %>%
      group_by(Treatment.Group) %>%
      summarise(count = n()) %>%
      arrange(desc(count))

In [ ]:
sdy524_antibody.df %>%
      group_by(Visit_Num) %>%
      summarise(count = n()) %>%
      arrange(desc(count))

In [ ]:
sdy524_subj <- as.matrix(read.csv("data/SDY524/SDY524_ITN027AI.DataSet.ADSTAND.csv",sep=","))

In [ ]:
sdy524_subj.df <- data.frame(sdy524_subj)

In [ ]:
colnames(sdy524_subj.df)

In [ ]:
colnames(sdy524_antibody.df)

In [ ]:
subject_df <- unique(sdy524_subj.df[,c ('ImmPort.Accession',
                                 'Participant_ID',
                                 'Age_Group',
                                 'Age_years',
                                 'Race',
                                 'Sex',
                                 'Ethnicity')])

In [ ]:
sdy524_df <- left_join(subject_df, sdy524_antibody.df, by = "ImmPort.Accession")

In [ ]:
any(duplicated(names(sdy524_df)))
names(sdy524_df)[duplicated(names(sdy524_df))]

In [ ]:
dim(sdy524_df)

In [ ]:
head(sdy524_df)

In [ ]:
sdy524_df %>%
      group_by(Age_Group) %>%
      summarise(count = n()) %>%
      arrange(desc(count))

In [ ]:
sdy524_df %>%
      group_by(Visit_Num) %>%
      summarise(count = n()) %>%
      arrange(desc(count))

In [ ]:
## baseline (-1)
tidy_524_v_m1 <- aa_prepare(
  sdy524_df,
  accession_col     = "ImmPort.Accession",
  sex_col           = "Sex",
  age_group_col     = "Age_Group",
  age_years_col     = "Age_years",
  analyte_name_col  = "AutoAB",
  analyte_value_col = "Value",
  visit_cols        = c("Visit_Num","Visitnum","Visit"),
  restrict_visits   = -1
)
hm_524_v_m1 <- aa_heatmaps(tidy_524_v_m1, scale_mode = "column", k_rows = 4, k_cols = 2)

In [ ]:
hm_524_v_m1$hm_ward

In [ ]:
hm_524_v_m1$hm_kmeans

In [ ]:
## day 21
tidy_524_v21 <- aa_prepare(
  sdy524_df,
  accession_col     = "ImmPort.Accession",
  sex_col           = "Sex",
  age_group_col     = "Age_Group",
  age_years_col     = "Age_years",
  analyte_name_col  = "AutoAB",
  analyte_value_col = "Value",
  visit_cols        = c("Visit_Num","Visitnum","Visit"),
  restrict_visits   = 21
)
hm_524_v21 <- aa_heatmaps(tidy_524_v21, scale_mode = "column", k_rows = 4, k_cols = 2)

In [ ]:
hm_524_v21$hm_ward

In [ ]:
hm_524_v21$hm_kmeans

In [ ]:
## day 42
tidy_524_v42 <- aa_prepare(
  sdy524_df,
  accession_col     = "ImmPort.Accession",
  sex_col           = "Sex",
  age_group_col     = "Age_Group",
  age_years_col     = "Age_years",
  analyte_name_col  = "AutoAB",
  analyte_value_col = "Value",
  visit_cols        = c("Visit_Num","Visitnum","Visit"),
  restrict_visits   = 42
)
hm_524_v42 <- aa_heatmaps(tidy_524_v42, scale_mode = "column", k_rows = 4, k_cols = 2)

In [ ]:
hm_524_v42$hm_ward

In [ ]:
hm_524_v42$hm_kmeans

**ImmPort SDY569**

In [ ]:
getwd()

In [ ]:
sdy569_antibody <- as.matrix(read.csv("data/SDY569/aa_569.csv",sep=","))
sdy569_antibody.df <- data.frame(sdy569_antibody)
head(sdy569_antibody.df)

In [ ]:
sdy569_subj <- as.matrix(read.csv("data/SDY569/demo_569.csv",sep=","))
sdy569_subj.df <- data.frame(sdy569_subj)
head(sdy569_subj.df)

In [ ]:
subject_df <- unique(sdy569_subj.df[, c(
  "accession",
  "study_id",
  "sex",
  "year_of_birth",
  "cohort_group",
  "race",
  "ethnicity",
  "day_0_date",
  "date_of_t1dm_diagnosis"
)])


In [ ]:
install.packages("tidyverse")
install.packages("lubridate")
library("tidyverse")
library("lubridate")

In [ ]:
sdy569_subj.df <- subject_df %>%
  mutate(
    day_0_date = as.Date(day_0_date),
    age_years = year(day_0_date) - as.numeric(year_of_birth),
    visit = -1,
    age_group = aa_age_to_group(age_years)
  )

In [ ]:
# ---- Ensure antibody dataframe has Accession for join
sdy569_antibody.df$accession <- as.character(sdy569_antibody.df$accession)

In [ ]:
# ---- Left join subject + antibody data
sdy569_df <- left_join(sdy569_subj.df, sdy569_antibody.df, by = "accession")

In [ ]:
colnames(sdy569_df)

In [ ]:
# ---- rename to match expectations for aa_prepare
sdy569_df <- sdy569_df %>%
  rename(
    Accession = accession,
    Sex = sex,
    Age_years = age_years,
    Age_Group = age_group,
    Visit = visit,
    Baseline_Height_cm = baseline_height_cm,
    Baseline_Weight_kg = baseline_weight_kg,
    Baseline_BMI_kg_m2 = baseline_bmi_kg_m_2
  )


In [ ]:
tidy_569 <- aa_prepare(
  sdy569_df,
  accession_col   = "Accession",
  sex_col         = "Sex",
  age_years_col   = "Age_years",
  age_group_col   = "Age_Group",
  analyte_cols    = c("gad65", "ia_2ic", "miaa"),  # adjust if different
  visit_cols      = c("visit"),
  restrict_visits = -1  # Or remove if not relevant
)

In [ ]:
hm_569 <- aa_heatmaps(
  tidy_569,
  scale_mode = "column",
  k_rows = 4,
  k_cols = 2
)

In [ ]:
# View plots
hm_569$hm_kmeans

In [ ]:
hm_569$hm_ward

**ImmPort SDY1625**

In [ ]:
# Load antibody results (already done earlier)
getwd()

In [ ]:
# Load demographics
sdy1625_demo <- as.matrix(read.csv("data/SDY1625/SDY1625_demo.csv",sep=","))
sdy1625_demo.df <- data.frame (sdy1625_demo)
colnames(sdy1625_demo.df)

In [ ]:
# Load autoantibody results
sdy1625_ab <- as.matrix(read.csv("data/SDY1625/SDY1625_elisa_result.csv",sep=","))
sdy1625_ab.df <- data.frame(sdy1625_ab)
head(sdy1625_ab.df)

In [ ]:
library(dplyr)
library(lubridate)

# Clean demographic information
sdy1625_demo.df <- sdy1625_demo.df %>%
  mutate(
    Accession = as.character(subjectAccession),
    Age_years = maxSubjectAge,
    Age_Group = aa_age_to_group(maxSubjectAge),
    Sex = as.character(gender),
    Visit = -1
  )


In [ ]:
# Clean elisa information
sdy1625_ab.df <- sdy1625_ab.df %>%
  mutate(
    Accession = as.character(SUBJECT_ACCESSION),
    Property = trimws(ANALYTE_REPORTED),
    Value_raw = trimws(VALUE_REPORTED),
    Value = suppressWarnings(as.numeric(Value_raw))
  )


In [ ]:
sdy1625_df <- left_join(sdy1625_ab.df, sdy1625_demo.df, by = "Accession")

In [ ]:
tidy_1625 <- aa_prepare(
  df                = sdy1625_df,
  accession_col     = "Accession",
  sex_col           = "Sex",
  age_years_col     = "Age_years",
  age_group_col     = "Age_Group",      # optional if age_years_col is given
  analyte_name_col  = "Property",       # from ELISA data
  analyte_value_col = "Value",          # from ELISA data
  visit_cols        = c("Visit"),
  restrict_visits   = -1
)


In [ ]:
# new function for renaming
ab_name_map <- c(
  "GAD Autoantibody"           = "GAD65",
  "IA-2 Autoantibody"          = "IA_2ic",
  "ZnT8 Autoantibody"          = "Zn_T8",
  "MIAA Autoantibody"          = "MIAA",
  "IA2ic"                      = "IA_2ic",  # in case of naming drift
  "ZNT8A"                      = "Zn_T8",   # ImmPort sometimes uses this
  "GADA"                       = "GAD65",
  "MIAA"                       = "MIAA"
)


In [ ]:
tidy_1625 <- tidy_1625 %>%
  mutate(Property = recode(Property, !!!ab_name_map))


In [ ]:
sum(is.nan(tidy_1625$Value))  # How many NaNs?

In [ ]:
tidy_1625 <- tidy_1625 %>%
  mutate(Value = ifelse(is.nan(Value), NA, Value))


In [ ]:
summary(tidy_1625$Value)
sum(is.na(tidy_1625$Value))
sum(is.nan(tidy_1625$Value))
sum(is.infinite(tidy_1625$Value))


In [ ]:
tidy_1625 <- tidy_1625 %>%
  mutate(Value = ifelse(is.na(Value), 0, as.numeric(Value)))


In [ ]:
hm_1625 <- aa_heatmaps(tidy_1625, 
                       scale_mode = "column", 
                       k_rows = 4, 
                       k_cols = 2)


In [ ]:
hm_1625$hm_kmeans

In [ ]:

# Show plots
hm_1625$hm_ward


**ImmPort SDY797**

In [ ]:
# Load demographics
getwd()
sdy797_demo <- as.matrix(read.csv("data/SDY797/SDY797_demo.csv",sep=","))
sdy797_demo.df <- data.frame (sdy797_demo)
colnames(sdy797_demo.df)
head(sdy797_demo.df)

In [ ]:
# Load autoantibody results
sdy797_aa <- as.matrix(read.csv("data/SDY797/sdy797_aa.csv",sep=","))
sdy797_aa.df <- data.frame(sdy797_aa)
colnames(sdy797_aa.df)
head(sdy797_aa.df)

In [ ]:
library(dplyr)
library(tidyr)
library(stringr)

# Your raw input
df <- sdy797_aa.df

# 1. Select antibody columns and rename to Property-friendly names
sdy797_aa_clean.df <- df %>%
  rename(Accession = ImmPort_Accession) %>%
  mutate(Study = "SDY797") %>%
  select(
    Accession,
    GAD_65_Result_less_than20_Negative,
    IA.2ic_Result_less_than5_Negative,
    MIAA_Result_less_than0.010_Negative,
    ZnT8_Result_less_than0.030_Negative,
    ICA_Result_less_than10_Negative,
    Study
  ) %>%
  pivot_longer(
    cols = -c(Accession, Study),
    names_to = "Property",
    values_to = "Value"
  ) %>%
  mutate(
    # Convert "Positive"/"Negative" to 1/0
    Value = case_when(
      tolower(Value) == "positive" ~ 1,
      tolower(Value) == "negative" ~ 0,
      TRUE ~ NA_real_
    ),
    # Harmonize property names
    Property = case_when(
      str_detect(Property, "GAD") ~ "GAD65",
      str_detect(Property, "IA.2ic") ~ "IA_2ic",
      str_detect(Property, "MIAA") ~ "MIAA",
      str_detect(Property, "ZnT8") ~ "Zn_T8",
      str_detect(Property, "ICA") ~ "ICA",
      TRUE ~ Property
    )
  ) %>%
  select(Accession, Property, Value, Study)

# Preview
print(head(sdy797_aa_clean.df, 10))


In [ ]:
library(dplyr)
library(lubridate)

# Clean demographic information
sdy797_demo.df <- sdy797_demo.df %>%
  mutate(
    Accession = as.character(subjectAccession),
    Age_years = maxSubjectAge,
    Age_Group = aa_age_to_group(maxSubjectAge),
    Sex = as.character(gender),
    Visit = -1,
    Study = as.character(studyAccession)
  )
head(sdy797_demo.df)

In [ ]:
sdy797_df <- left_join(sdy797_aa_clean.df, sdy797_demo.df, by = "Accession")

In [ ]:
head(sdy797_df)

In [ ]:
tidy_797 <- aa_prepare(
  df                = sdy797_df,
  accession_col     = "Accession",
  sex_col           = "Sex",
  age_years_col     = "Age_years",
  age_group_col     = "Age_Group",
  analyte_name_col  = "Property",
  analyte_value_col = "Value",
  visit_cols        = c("Visit"),
  restrict_visits   = -1
)


In [ ]:
hm_797 <- aa_heatmaps(tidy_797, 
                       scale_mode = "column", 
                       k_rows = 4, 
                       k_cols = 2)


In [ ]:
hm_797$hm_kmeans

In [ ]:
hm_797$hm_ward

In [ ]:
head(tidy_797)

In [ ]:
colnames(tidy_797)
colnames(tidy_524_v_m1)
colnames(tidy_1737_v1)
colnames(tidy_569)
colnames(tidy_1625)

In [ ]:
tidy_797  <- tidy_797 %>% mutate(Study="SDY797")
tidy_1737 <- tidy_1737_v1 %>% mutate(Study = "SDY1737")
tidy_524  <- tidy_524_v_m1  %>% mutate(Study = "SDY524")
tidy_569  <- tidy_569  %>% mutate(Study = "SDY569")
tidy_1625 <- tidy_1625 %>% mutate(Study = "SDY1625")


In [ ]:
head(tidy_797)
head(tidy_1737)
head(tidy_524)
head(tidy_569)
head(tidy_1625)

In [ ]:
harmonize_property_names <- function(df) {
  df %>%
    mutate(Property = case_when(
      tolower(Property) %in% c("gad", "gad65", "gad_65")     ~ "GAD65",
      tolower(Property) %in% c("ia2", "ia_2ic", "ia-2ic")     ~ "IA_2ic",
      tolower(Property) %in% c("miaa")                        ~ "MIAA",
      tolower(Property) %in% c("znt8", "zn_t8", "znt-8")      ~ "Zn_T8",
      TRUE ~ Property  # leave as-is for unknowns
    ))
}


In [ ]:
tidy_all <- bind_rows(tidy_797,
                      tidy_1737,
                      tidy_524,
                      tidy_569,
                      tidy_1625)

In [ ]:
tidy_all <- tidy_all %>% harmonize_property_names()


In [ ]:
str(tidy_all)
summary(tidy_all)


In [ ]:
install.packages("BiocManager")
BiocManager::install("tidyHeatmap")


In [ ]:
tidy_all <- tidy_all %>%
  mutate(Value = as.numeric(Value))  # Forces coercion from character
summary(tidy_all)
dim(tidy_all)

In [ ]:
any(is.na(tidy_all$Value)) # should be FALSE
length(unique(tidy_all$Accession)) > 1 # TRUE
length(unique(tidy_all$Property)) > 1 # TRUE


In [ ]:
sum(is.na(tidy_all$Value))


In [ ]:
tidy_all <- tidy_all %>%
  mutate(Value = ifelse(is.na(Value), 0, Value))


In [ ]:
any(is.na(tidy_all$Value))  # should now return FALSE


In [ ]:
all_hm <- aa_heatmaps(tidy_all, scale_mode = "column", k_rows = 4, k_cols = 2)

In [ ]:
all_hm$hm_ward

In [ ]:
all_hm$hm_kmeans

**save for further work in federated learning with convolution neural networks**

In [ ]:
write.csv(tidy_all,   "data/tidy_all.csv",   row.names = FALSE)
write.csv(tidy_797,   "data/tidy_797.csv",   row.names = FALSE)
write.csv(tidy_1625,  "data/tidy_1625.csv",  row.names = FALSE)
write.csv(tidy_524,   "data/tidy_524.csv",   row.names = FALSE)
write.csv(tidy_569,   "data/tidy_569.csv",   row.names = FALSE)
write.csv(tidy_1737,  "data/tidy_1737.csv",  row.names = FALSE)


In [ ]:
# Pick the autoantibody you want
antibody_of_interest <- "GAD65"

tidy_single_gad65 <- tidy_all %>%
  filter(Property == "GAD65")

gad65_hm <- aa_heatmaps(tidy_single_gad65, scale_mode = "column", k_rows = 4, k_cols = 2)

In [ ]:
gad65_hm$hm_ward

In [ ]:
gad65_hm$hm_kmeans

In [ ]:
tidy_all_clean <- tidy_all %>%
  group_by(Study, Property) %>%
  mutate(Value = scale(Value)[, 1]) %>%  # z-score within group
  ungroup()
summary(tidy_all_clean)
dim(tidy_all_clean)

In [ ]:
colnames(tidy_all_clean)
tidy_all_clean <- tidy_all_clean %>%
  filter(!is.na(Accession), !is.na(Property), !is.na(Value))
dim(tidy_all_clean)

In [ ]:
tidy_all_clean <- tidy_all_clean %>%
  group_by(Study, Property) %>%
  mutate(Value = scale(Value)[, 1]) %>%
  ungroup()


In [ ]:
dupes <- tidy_all_clean %>%
  count(Accession, Property) %>%
  filter(n > 1)

if (nrow(dupes) > 0) print(dupes)


In [ ]:
any(is.na(tidy_all_clean$Value)) # should be FALSE
length(unique(tidy_all_clean$Accession)) > 1 # TRUE
length(unique(tidy_all_clean$Property)) > 1 # TRUE


In [ ]:
hm_scaled <- aa_heatmaps(tidy_all_clean, k_rows = 4, k_cols = 2, scale_mode="column")


In [ ]:
hm_scaled$hm_kmeans